# FASTopic PPD Notebook

Pipeline aligned with ECTRM datasets and `.mat` outputs.


## 1) Install dependencies (Kaggle-safe)


In [1]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str, no_deps: bool = False):
    if importlib.util.find_spec(import_name) is None:
        cmd = [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade-strategy',
            'only-if-needed',
            '--no-cache-dir',
        ]
        if no_deps:
            cmd.append('--no-deps')
        cmd.append(pip_spec)
        print('Installing', pip_spec)
        subprocess.check_call(cmd)


# Important: we do NOT install/upgrade torch, CUDA, or GPU drivers.
# Kaggle's existing GPU stack is reused as-is.
ensure_package('scipy', 'scipy')
ensure_package('yaml', 'pyyaml')
ensure_package('sentence_transformers', 'sentence-transformers')
ensure_package('fastopic', 'fastopic==1.0.1', no_deps=True)

print('Dependency check OK (torch/cuda untouched)')


  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [716 lines of output]
      C:\Users\lesartisansverts\AppData\Local\Temp\pip-build-env-mkl_thui\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\lesartisansverts\AppData\Local\Temp\pip-build-env-mkl_thui\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\gensim
      copying gensim\downloader.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\interfaces.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\matutils.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\nosy.py -> build\lib.win-amd64-

In [3]:
!pip install numpy

  Using cached numpy-2.4.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Using cached numpy-2.4.2-cp314-cp314-win_amd64.whl (12.4 MB)



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Imports and env


In [4]:
import os
import random
from pathlib import Path

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
import yaml
from sklearn import metrics
from fastopic import FASTopic

print('Python:', __import__('sys').version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


ModuleNotFoundError: No module named 'scipy'

TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz',
    'test_bow.npz',
    'train_labels.txt',
    'test_labels.txt',
    'vocab.txt',
    'word_embeddings.npz',
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 6):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


EXPLICIT_KAGGLE_CANDIDATES = {
    '20NG': [
        '/kaggle/input/ectrm-source-data-20ng',
        '/kaggle/input/kaggleinputectrm-source-data-20ng',
        '/kaggle/input/datasets/snlosnl/kaggleinputectrm-source-data-20ng',
    ],
    'AGNews': [
        '/kaggle/input/ECTRM_source\\data\\AGNews',
        '/kaggle/input/ectrm-sourcedataagnews',
        '/kaggle/input/ectrm-source-data-agnews',
        '/kaggle/input/datasets/snlosnl/ectrm-sourcedataagnews',
    ],
    'IMDB': [
        '/kaggle/input/ectrm-sourcedataIMDB',
        '/kaggle/input/ectrm-sourcedataimdb',
        '/kaggle/input/ectrm-source-data-imdb',
        '/kaggle/input/datasets/snlosnl/ectrm-sourcedataimdb',
    ],
    'YahooAnswer': [
        '/kaggle/input/ECTRM-YahooAnswer',
        '/kaggle/input/ectrm-yahooanswer',
        '/kaggle/input/ectrm-source-data-yahooanswer',
        '/kaggle/input/datasets/snlosnl/ectrm-yahooanswer',
    ],
}


def first_valid_path(candidates):
    for pstr in candidates:
        p = Path(pstr)
        if has_required_files(p):
            return p
    return None


def discover_dataset_dirs():
    found = {}

    # 1) Explicit Kaggle candidates first (exactly matching your mounted datasets).
    if IS_KAGGLE:
        for ds, candidates in EXPLICIT_KAGGLE_CANDIDATES.items():
            p = first_valid_path(candidates)
            if p is not None:
                found[ds] = p

    # 2) Local canonical layout.
    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p) and ds not in found:
            found[ds] = p

    # 3) Generic scan fallback.
    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand

    return found


DATASET_DIRS = discover_dataset_dirs()
print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)

if IS_KAGGLE and Path('/kaggle/input').exists():
    print('\nTop-level /kaggle/input dirs:')
    for d in sorted([x.name for x in Path('/kaggle/input').iterdir() if x.is_dir()]):
        print('-', d)

print('\nResolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


In [ ]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz', 'test_bow.npz', 'train_labels.txt', 'test_labels.txt', 'vocab.txt', 'word_embeddings.npz'
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p):
            found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand
    return found


DATASET_DIRS = discover_dataset_dirs()
print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)
print('\nResolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


## 4) Config and helpers


In [ ]:
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / 'FASTopic') if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / 'FASTopic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

# FASTopic-specific defaults (phase 2 tuning).
# Important: we intentionally do NOT reuse ECRTM batch_size=200 for low_memory_batch_size,
# because FASTopic docs warn that too small low_memory_batch_size hurts quality.
FASTOPIC_DEFAULTS = {
    'epochs': 200,
    'learning_rate': 0.002,
    'low_memory': True,
    'normalize_embeddings': True,
    'doc_embed_model': 'all-MiniLM-L6-v2',
    'min_low_memory_batch_size': 2000,
    'max_low_memory_batch_size': 4000,
    'dt_alpha_default': 10.0,
    'tw_alpha': 2.0,
    'theta_temp': 1.0,
}

# Options: 'auto' | 'text' | 'bow_avg'
DOC_EMBEDDING_MODE = 'auto'

# Good default split for your datasets (short texts -> bow_avg, long texts -> text).
DOC_MODE_BY_DATASET = {
    '20NG': 'text',
    'IMDB': 'text',
    'AGNews': 'bow_avg',
    'YahooAnswer': 'bow_avg',
}

# FASTopic README suggests higher DT_alpha when topics are repetitive.
DT_ALPHA_BY_DATASET = {
    '20NG': 8.0,
    'IMDB': 8.0,
    'AGNews': 12.0,
    'YahooAnswer': 12.0,
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_cfg(path):
    if path is None or (not Path(path).exists()):
        return {}
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def discover_cfg_paths():
    cfg = {}

    local_cfg_roots = [
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'ECRTM' / 'configs' / 'model',
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'configs' / 'model',
    ]

    for root in local_cfg_roots:
        for ds in TARGET_DATASETS:
            p = root / f'ECRTM_{ds}.yaml'
            if p.exists():
                cfg[ds] = p

    for ds in TARGET_DATASETS:
        if ds in cfg:
            continue
        name = f'ECRTM_{ds}.yaml'
        for base in [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]:
            if not base.exists():
                continue
            found = None
            for r, _, files in os.walk(base):
                if name in files:
                    found = Path(r) / name
                    break
            if found is not None:
                cfg[ds] = found
                break

    return cfg


CONFIGS = discover_cfg_paths()
for ds in TARGET_DATASETS:
    print(ds, 'cfg:', CONFIGS.get(ds))


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path, vocab_size=None):
    try:
        arr = sp.load_npz(path).astype(np.float32).toarray()
        if vocab_size is not None and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr
    except Exception:
        data = np.load(path)
        if isinstance(data, np.lib.npyio.NpzFile):
            arr = data[list(data.keys())[0]]
        else:
            arr = data
        arr = np.asarray(arr, dtype=np.float32)
        if vocab_size is not None:
            if arr.ndim == 1:
                emb_dim = arr.size // vocab_size
                arr = arr[: vocab_size * emb_dim].reshape(vocab_size, emb_dim)
            elif arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
                arr = arr.T
        return arr.astype(np.float32)


def load_texts_if_exists(path: Path):
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]


def average_token_length(texts):
    if texts is None or len(texts) == 0:
        return 0.0
    return float(np.mean([len(t.split()) for t in texts]))


def resolve_doc_mode(dataset, train_docs):
    if DOC_EMBEDDING_MODE in ('text', 'bow_avg'):
        return DOC_EMBEDDING_MODE

    # 'auto': dataset-specific rule first, then length heuristic fallback.
    mode = DOC_MODE_BY_DATASET.get(dataset)
    if mode is not None:
        return mode

    avg_len = average_token_length(train_docs)
    return 'text' if avg_len >= 30 else 'bow_avg'


def resolve_low_memory_batch_size(n_docs):
    min_bs = int(FASTOPIC_DEFAULTS['min_low_memory_batch_size'])
    max_bs = int(FASTOPIC_DEFAULTS['max_low_memory_batch_size'])
    if n_docs <= min_bs:
        return int(n_docs)
    candidate = max(min_bs, int(0.2 * n_docs))
    return int(min(candidate, max_bs))


def build_doc_embeddings_from_bow(bow_csr, word_emb, normalize=True, eps=1e-12, chunk_size=20000):
    n_docs = bow_csr.shape[0]
    emb_dim = word_emb.shape[1]
    out = np.zeros((n_docs, emb_dim), dtype=np.float32)

    lengths = np.asarray(bow_csr.sum(axis=1), dtype=np.float32).reshape(-1, 1)
    lengths[lengths == 0] = 1.0

    for start in range(0, n_docs, chunk_size):
        end = min(start + chunk_size, n_docs)
        part = bow_csr[start:end].dot(word_emb)
        out[start:end] = np.asarray(part, dtype=np.float32) / lengths[start:end]

    if normalize:
        norms = np.linalg.norm(out, axis=1, keepdims=True) + eps
        out = out / norms

    return out


class FixedPreprocess:
    def __init__(self, train_bow, vocab):
        self._train_bow = train_bow
        self._vocab = vocab

    def preprocess(self, docs):
        if len(docs) != self._train_bow.shape[0]:
            raise ValueError('docs length must match train_bow rows')
        return {'train_bow': self._train_bow, 'vocab': self._vocab}


class PassthroughDocEmbedder:
    def encode(self, docs, show_progress_bar=False, normalize_embeddings=False):
        raise RuntimeError('encode should not be called when preset_doc_embeddings is provided')


def infer_theta_from_doc_embeddings(model, doc_embeddings_np):
    doc_embeddings = torch.as_tensor(doc_embeddings_np, dtype=torch.float32)
    if not model.low_memory:
        doc_embeddings = doc_embeddings.to(model.device)
    return np.asarray(model.transform(doc_embeddings=doc_embeddings), dtype=np.float32)


def build_bow_avg_doc_embeddings(data_dir, train_bow, test_bow, vocab_size):
    word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=vocab_size)
    train_doc_emb = build_doc_embeddings_from_bow(
        train_bow,
        word_emb,
        normalize=FASTOPIC_DEFAULTS['normalize_embeddings'],
    )
    test_doc_emb = build_doc_embeddings_from_bow(
        test_bow,
        word_emb,
        normalize=FASTOPIC_DEFAULTS['normalize_embeddings'],
    )
    return train_doc_emb, test_doc_emb


def make_fastopic_model(K, preprocess, doc_embed_model, device, low_memory_batch_size, dt_alpha, tw_alpha, theta_temp):
    return FASTopic(
        num_topics=K,
        preprocess=preprocess,
        doc_embed_model=doc_embed_model,
        device=device,
        low_memory=bool(FASTOPIC_DEFAULTS['low_memory']),
        low_memory_batch_size=low_memory_batch_size if FASTOPIC_DEFAULTS['low_memory'] else None,
        normalize_embeddings=bool(FASTOPIC_DEFAULTS['normalize_embeddings']),
        DT_alpha=dt_alpha,
        TW_alpha=tw_alpha,
        theta_temp=theta_temp,
        verbose=False,
    )


def train_one_fastopic(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} not found. Available: {list(DATASET_DIRS.keys())}')

    set_seed(seed)
    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS.get(dataset))

    # Keep FASTopic-specific defaults; allow optional overrides via custom keys.
    epochs = int(cfg.get('fastopic_epochs', FASTOPIC_DEFAULTS['epochs']))
    learning_rate = float(cfg.get('fastopic_learning_rate', FASTOPIC_DEFAULTS['learning_rate']))

    train_bow = load_bow_csr(data_dir / 'train_bow.npz')
    test_bow = load_bow_csr(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    vocab_size = train_bow.shape[1]

    preprocess = FixedPreprocess(train_bow=train_bow, vocab=vocab)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    train_docs = load_texts_if_exists(data_dir / 'train_texts.txt')
    test_docs = load_texts_if_exists(data_dir / 'test_texts.txt')

    selected_mode = resolve_doc_mode(dataset, train_docs)
    can_use_text = (
        train_docs is not None
        and test_docs is not None
        and len(train_docs) == train_bow.shape[0]
        and len(test_docs) == test_bow.shape[0]
    )
    use_text_mode = (selected_mode == 'text' and can_use_text)

    test_doc_emb = None
    if use_text_mode:
        docs_for_fit = train_docs
        preset_doc_embeddings = None
        doc_embed_model = FASTOPIC_DEFAULTS['doc_embed_model']
        avg_len = average_token_length(train_docs)
        print(f'[FASTopic] {dataset} K={K}: text mode (avg train tokens={avg_len:.1f})')
    else:
        train_doc_emb, test_doc_emb = build_bow_avg_doc_embeddings(data_dir, train_bow, test_bow, vocab_size)
        docs_for_fit = [''] * train_bow.shape[0]
        preset_doc_embeddings = train_doc_emb
        doc_embed_model = PassthroughDocEmbedder()
        print(f'[FASTopic] {dataset} K={K}: bow_avg mode')

    low_memory_batch_size = resolve_low_memory_batch_size(train_bow.shape[0])
    dt_alpha = float(cfg.get('fastopic_dt_alpha', DT_ALPHA_BY_DATASET.get(dataset, FASTOPIC_DEFAULTS['dt_alpha_default'])))
    tw_alpha = float(cfg.get('fastopic_tw_alpha', FASTOPIC_DEFAULTS['tw_alpha']))
    theta_temp = float(cfg.get('fastopic_theta_temp', FASTOPIC_DEFAULTS['theta_temp']))

    print(
        f"[FASTopic] {dataset} K={K} epochs={epochs} lr={learning_rate} "
        f"low_memory_batch_size={low_memory_batch_size} DT_alpha={dt_alpha}"
    )

    def _fit_current_setup():
        model = make_fastopic_model(
            K=K,
            preprocess=preprocess,
            doc_embed_model=doc_embed_model,
            device=device,
            low_memory_batch_size=low_memory_batch_size,
            dt_alpha=dt_alpha,
            tw_alpha=tw_alpha,
            theta_temp=theta_temp,
        )
        model.fit_transform(
            docs=docs_for_fit,
            epochs=epochs,
            learning_rate=learning_rate,
            preset_doc_embeddings=preset_doc_embeddings,
        )
        return model

    try:
        model = _fit_current_setup()
    except Exception as exc:
        if use_text_mode:
            # Robust fallback when sentence-transformer model download/runtime fails.
            print(f'[FASTopic] text mode failed for {dataset} K={K}: {repr(exc)}')
            print('[FASTopic] fallback to bow_avg mode')
            train_doc_emb, test_doc_emb = build_bow_avg_doc_embeddings(data_dir, train_bow, test_bow, vocab_size)
            docs_for_fit = [''] * train_bow.shape[0]
            preset_doc_embeddings = train_doc_emb
            doc_embed_model = PassthroughDocEmbedder()
            use_text_mode = False
            model = _fit_current_setup()
        else:
            raise

    beta = np.asarray(model.get_beta(), dtype=np.float32)
    train_theta = np.asarray(model.train_theta, dtype=np.float32)

    if use_text_mode:
        test_theta = np.asarray(model.transform(docs=test_docs), dtype=np.float32)
    else:
        test_theta = infer_theta_from_doc_embeddings(model, test_doc_emb)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{seed}.mat'

    scipy.io.savemat(
        str(out_path),
        {
            'beta': beta,
            'train_theta': train_theta,
            'test_theta': test_theta,
            'traintheta': train_theta,
            'testtheta': test_theta,
        },
    )

    print('Saved:', out_path)
    print('beta', beta.shape, 'train_theta', train_theta.shape, 'test_theta', test_theta.shape)
    return out_path


In [ ]:
def train_one_fastopic(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} not found. Available: {list(DATASET_DIRS.keys())}')

    set_seed(seed)
    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS.get(dataset))

    # Keep FASTopic-specific defaults; allow optional overrides via custom keys.
    epochs = int(cfg.get('fastopic_epochs', FASTOPIC_DEFAULTS['epochs']))
    learning_rate = float(cfg.get('fastopic_learning_rate', FASTOPIC_DEFAULTS['learning_rate']))

    train_bow = load_bow_csr(data_dir / 'train_bow.npz')
    test_bow = load_bow_csr(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    vocab_size = train_bow.shape[1]

    preprocess = FixedPreprocess(train_bow=train_bow, vocab=vocab)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    train_docs = load_texts_if_exists(data_dir / 'train_texts.txt')
    test_docs = load_texts_if_exists(data_dir / 'test_texts.txt')

    selected_mode = resolve_doc_mode(dataset, train_docs)
    can_use_text = (
        train_docs is not None
        and test_docs is not None
        and len(train_docs) == train_bow.shape[0]
        and len(test_docs) == test_bow.shape[0]
    )
    use_text_mode = (selected_mode == 'text' and can_use_text)

    if use_text_mode:
        docs_for_fit = train_docs
        preset_doc_embeddings = None
        doc_embed_model = FASTOPIC_DEFAULTS['doc_embed_model']
        avg_len = average_token_length(train_docs)
        print(f'[FASTopic] {dataset} K={K}: text mode (avg train tokens={avg_len:.1f})')
    else:
        word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=vocab_size)
        train_doc_emb = build_doc_embeddings_from_bow(
            train_bow,
            word_emb,
            normalize=FASTOPIC_DEFAULTS['normalize_embeddings'],
        )
        test_doc_emb = build_doc_embeddings_from_bow(
            test_bow,
            word_emb,
            normalize=FASTOPIC_DEFAULTS['normalize_embeddings'],
        )
        docs_for_fit = [''] * train_bow.shape[0]
        preset_doc_embeddings = train_doc_emb
        doc_embed_model = PassthroughDocEmbedder()
        print(f'[FASTopic] {dataset} K={K}: bow_avg mode')

    low_memory_batch_size = resolve_low_memory_batch_size(train_bow.shape[0])
    dt_alpha = float(cfg.get('fastopic_dt_alpha', DT_ALPHA_BY_DATASET.get(dataset, FASTOPIC_DEFAULTS['dt_alpha_default'])))
    tw_alpha = float(cfg.get('fastopic_tw_alpha', FASTOPIC_DEFAULTS['tw_alpha']))
    theta_temp = float(cfg.get('fastopic_theta_temp', FASTOPIC_DEFAULTS['theta_temp']))

    print(
        f"[FASTopic] {dataset} K={K} epochs={epochs} lr={learning_rate} "
        f"low_memory_batch_size={low_memory_batch_size} DT_alpha={dt_alpha}"
    )

    model = FASTopic(
        num_topics=K,
        preprocess=preprocess,
        doc_embed_model=doc_embed_model,
        device=device,
        low_memory=bool(FASTOPIC_DEFAULTS['low_memory']),
        low_memory_batch_size=low_memory_batch_size if FASTOPIC_DEFAULTS['low_memory'] else None,
        normalize_embeddings=bool(FASTOPIC_DEFAULTS['normalize_embeddings']),
        DT_alpha=dt_alpha,
        TW_alpha=tw_alpha,
        theta_temp=theta_temp,
        verbose=False,
    )

    model.fit_transform(
        docs=docs_for_fit,
        epochs=epochs,
        learning_rate=learning_rate,
        preset_doc_embeddings=preset_doc_embeddings,
    )

    beta = np.asarray(model.get_beta(), dtype=np.float32)
    train_theta = np.asarray(model.train_theta, dtype=np.float32)

    if use_text_mode:
        test_theta = np.asarray(model.transform(docs=test_docs), dtype=np.float32)
    else:
        test_theta = infer_theta_from_doc_embeddings(model, test_doc_emb)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{seed}.mat'

    scipy.io.savemat(
        str(out_path),
        {
            'beta': beta,
            'train_theta': train_theta,
            'test_theta': test_theta,
            'traintheta': train_theta,
            'testtheta': test_theta,
        },
    )

    print('Saved:', out_path)
    print('beta', beta.shape, 'train_theta', train_theta.shape, 'test_theta', test_theta.shape)
    return out_path


## 6) Run


In [ ]:
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print('SKIP missing dataset:', dataset)
        continue
    for K in K_LIST:
        train_one_fastopic(dataset, K=K, seed=RANDOM_SEED)


## 7) Metrics and topics files


In [ ]:
import pandas as pd


def topic_diversity_from_beta(beta, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


rows = []
TOPN = 15
for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    ds_out = OUTPUT_DIR / dataset
    vocab = load_vocab(DATASET_DIRS[dataset] / 'vocab.txt')
    labels = np.loadtxt(DATASET_DIRS[dataset] / 'test_labels.txt', dtype=np.int32)

    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        rows.append({
            'model': 'FASTopic',
            'dataset': dataset,
            'K': K,
            'Purity': float(purity_score(y_true, y_pred)),
            'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            'TD_top15': float(topic_diversity_from_beta(beta, topn=15)),
            'TD_top25': float(topic_diversity_from_beta(beta, topn=25)),
        })

        txt_path = ds_out / f'topics_for_cv_{dataset}_FASTopic_K{K}_seed{RANDOM_SEED}.txt'
        with open(txt_path, 'w', encoding='utf-8') as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(' '.join(vocab[i] for i in top_idx) + '\n')
        print('Saved:', txt_path)

if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)
    csv_path = OUTPUT_DIR / 'FASTopic_metrics_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)
else:
    print('No FASTopic .mat found yet')

print('\\nFiles in OUTPUT_DIR:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print('-', p.relative_to(OUTPUT_DIR))
